In [2]:
# ==============================================================================
# ATIVIDADE: SISTEMA DE FILA DE EMERGÊNCIA HOSPITALAR
# ==============================================================================
# Estrutura de Dados
# Linguagem: Python
# Tema: Referências e estruturas dinâmicas.
# Aluno: Kauã de Sousa Franco da Costa
#
# Neste exercício será utilizada uma Lista Encadeada para representar uma fila de atendimento de emergência.
#
# A fila utiliza DUAS REGRAS DE PRIORIDADE:
#
# 1ª REGRA - CLASSIFICAÇÃO DE RISCO
#
# 🔴 VERMELHO -> maior prioridade
# 🟡 AMARELO  -> prioridade intermediária
# 🟢 VERDE    -> menor prioridade
#
# 2ª REGRA - IDADE
#
# Quando dois pacientes possuem a mesma classificação, o paciente mais velho terá prioridade.
#
# Caso dois pacientes possuam a mesma classificação E a mesma idade, será respeitada a ordem de chegada.
#
# Portanto:
#
# CLASSIFICAÇÃO -> IDADE -> ORDEM DE CHEGADA
#
# A fila será implementada utilizando nós e referências, sem utilizar uma lista comum do Python para organizar os pacientes.
# ==============================================================================
# PARTE 1: Classe Paciente
# ==============================================================================

class Paciente:
    """
    Classe responsável por armazenar as informações de cada paciente.

    Cada paciente possui:
    - nome;
    - idade;
    - classificação de risco;
    - nível de prioridade.
    """

    def __init__(self, nome: str, idade: int, classificacao: str):

        # Armazena o nome do paciente.
        self.nome = nome

        # Armazena a idade do paciente.
        self.idade = idade

        # Converte a classificação para letras maiúsculas.
        self.classificacao = classificacao.upper()

        # Define o nível de prioridade da classificação.
        #
        # Quanto menor o número, maior a prioridade.
        if self.classificacao == "VERMELHO":
            self.nivel = 1

        elif self.classificacao == "AMARELO":
            self.nivel = 2

        elif self.classificacao == "VERDE":
            self.nivel = 3

        else:
            # Caso seja informada uma classificação inválida,
            # o paciente será considerado verde.
            self.classificacao = "VERDE"
            self.nivel = 3


    def chave_prioridade(self):
        """
        Cria uma chave utilizada para comparar a prioridade
        entre dois pacientes.

        Primeiro é considerada a classificação.

        Se a classificação for igual, a idade será utilizada.

        O sinal negativo na idade faz com que uma idade maior
        tenha prioridade.
        """

        # Retorna:
        # 1º -> nível da classificação
        # 2º -> idade negativa
        return (self.nivel, -self.idade)


    def __repr__(self):
        """
        Representa o paciente em formato de texto.
        """

        # Define o símbolo visual da classificação.
        if self.classificacao == "VERMELHO":
            simbolo = "🔴"

        elif self.classificacao == "AMARELO":
            simbolo = "🟡"

        else:
            simbolo = "🟢"

        # Retorna os dados do paciente.
        return (
            f"{simbolo} [{self.nome}, {self.idade} anos - "
            f"{self.classificacao}]"
        )


# ==============================================================================
# PARTE 2: Classe Node
# ==============================================================================

class Node:
    """
    Estrutura básica de um nó da lista encadeada.

    Cada nó possui:
    - dado: contém um paciente;
    - proximo: referência para o próximo nó.
    """

    def __init__(self, paciente: Paciente):

        # Armazena o paciente dentro do nó.
        self.dado = paciente

        # Inicialmente o nó não aponta para nenhum outro nó.
        self.proximo = None


# ==============================================================================
# PARTE 3: Classe FilaEmergencia
# ==============================================================================

class FilaEmergencia:
    """
    Gerencia a fila de pacientes da emergência.

    A fila possui:

    inicio:
        Referência para o primeiro paciente.

    fim:
        Referência para o último paciente.

    tamanho:
        Quantidade de pacientes na fila.

    A posição de cada paciente é determinada por:
    1. Classificação de risco;
    2. Idade;
    3. Ordem de chegada.
    """

    def __init__(self):

        # No início a fila está vazia.
        self.inicio = None

        # Como não existem pacientes,
        # não existe um último nó.
        self.fim = None

        # Guarda a quantidade de pacientes.
        self.tamanho = 0


    def esta_vazia(self) -> bool:
        """
        Verifica se a fila está vazia.

        Retorna True caso o início da fila seja None.
        """

        return self.inicio is None


    def adicionar(self, nome: str, idade: int, classificacao: str):
        """
        Adiciona um paciente na fila.

        A posição do paciente será definida através de
        duas regras:

        1. Classificação de risco;
        2. Idade.

        Se classificação e idade forem iguais,
        o paciente que chegou primeiro continuará na frente.
        """

        # Cria o objeto paciente.
        novo_paciente = Paciente(
            nome,
            idade,
            classificacao
        )

        # Cria um novo nó para armazenar o paciente.
        novo_no = Node(novo_paciente)


        # ----------------------------------------------------------------------
        # CASO 1: FILA VAZIA
        # ----------------------------------------------------------------------

        if self.esta_vazia():

            # O novo nó passa a ser o primeiro da fila.
            self.inicio = novo_no

            # Como existe apenas um paciente,
            # ele também é o último.
            self.fim = novo_no

            # Atualiza a quantidade.
            self.tamanho += 1

            print(
                f"-> {novo_paciente.nome} entrou como primeiro "
                f"paciente da emergência."
            )

            return


        # ----------------------------------------------------------------------
        # CASO 2: INSERÇÃO COM PRIORIDADE
        # ----------------------------------------------------------------------
        #
        # Agora precisamos encontrar a posição correta do novo paciente.
        #
        # A comparação será feita através da chave de prioridade:
        #
        # (classificação, -idade)
        #
        # Exemplo:
        #
        # Vermelho, 70 anos -> (1, -70)
        # Vermelho, 40 anos -> (1, -40)
        #
        # O paciente de 70 anos terá prioridade.
        # ----------------------------------------------------------------------

        # Obtém a chave de prioridade do novo paciente.
        chave_novo = novo_paciente.chave_prioridade()


        # ----------------------------------------------------------------------
        # SUBCASO 2.1: NOVO PACIENTE ENTRA NO INÍCIO
        # ----------------------------------------------------------------------

        # Comparamos o novo paciente com o primeiro da fila.
        if chave_novo < self.inicio.dado.chave_prioridade():

            # O novo nó aponta para o antigo primeiro.
            novo_no.proximo = self.inicio

            # O novo nó passa a ser o início.
            self.inicio = novo_no

            # Atualiza o tamanho.
            self.tamanho += 1

            print(
                f"-> {novo_paciente.nome} "
                f"({novo_paciente.classificacao}, "
                f"{novo_paciente.idade} anos) entrou no início da fila."
            )

            return


        # ----------------------------------------------------------------------
        # SUBCASO 2.2: PROCURANDO A POSIÇÃO CORRETA
        # ----------------------------------------------------------------------

        # Começa pelo primeiro nó.
        atual = self.inicio


        # Percorremos a fila procurando onde o novo paciente
        # deve ser inserido.
        #
        # A condição utiliza <= para que, quando dois pacientes
        # tiverem exatamente a mesma classificação e idade,
        # o novo paciente fique DEPOIS do antigo.
        #
        # Dessa forma, a ordem de chegada é preservada.

        while (
            atual.proximo is not None
            and atual.proximo.dado.chave_prioridade() <= chave_novo
        ):

            # Avança para o próximo nó.
            atual = atual.proximo


        # ----------------------------------------------------------------------
        # INSERINDO O NOVO NÓ
        # ----------------------------------------------------------------------

        # O novo nó passa a apontar para o nó que vinha depois.
        novo_no.proximo = atual.proximo

        # O nó atual passa a apontar para o novo nó.
        atual.proximo = novo_no


        # ----------------------------------------------------------------------
        # ATUALIZANDO O FIM DA FILA
        # ----------------------------------------------------------------------

        # Se o novo nó não possui próximo,
        # significa que ele está no final.
        if novo_no.proximo is None:

            # Atualiza a referência fim.
            self.fim = novo_no


        # Atualiza o tamanho da fila.
        self.tamanho += 1

        print(
            f"-> {novo_paciente.nome} "
            f"({novo_paciente.classificacao}, "
            f"{novo_paciente.idade} anos) entrou na fila."
        )


    def atender(self):
        """
        Remove e retorna o primeiro paciente da fila.

        O primeiro paciente será sempre aquele que possui
        maior prioridade de acordo com as regras estabelecidas.
        """

        # Verifica se a fila está vazia.
        if self.esta_vazia():

            print(
                "\n⚠️ A emergência está vazia! "
                "Nenhum paciente aguardando."
            )

            return None


        # Guarda o paciente que será atendido.
        paciente_atendido = self.inicio.dado

        # O segundo nó passa a ser o primeiro.
        self.inicio = self.inicio.proximo

        # Diminui o tamanho da fila.
        self.tamanho -= 1


        # Caso a fila fique vazia,
        # o fim também deve ser None.
        if self.inicio is None:

            self.fim = None


        # Mostra quem foi atendido.
        print(
            f"\n🏥 Atendendo: {paciente_atendido}"
        )

        # Retorna o paciente atendido.
        return paciente_atendido


    def listar_espera(self):
        """
        Percorre a fila do início ao fim
        e exibe todos os pacientes.
        """

        print("\n--- 🏥 FILA DE ESPERA DA EMERGÊNCIA ---")


        # Verifica se não existem pacientes.
        if self.esta_vazia():

            print("Nenhum paciente aguardando.")
            print("----------------------------------------\n")

            return


        # Começa no primeiro nó.
        atual = self.inicio

        # Controla a posição do paciente.
        posicao = 1


        # Percorre todos os nós da fila.
        while atual is not None:

            # Exibe a posição e os dados do paciente.
            print(
                f"{posicao}º -> {atual.dado}"
            )

            # Avança para o próximo nó.
            atual = atual.proximo

            # Atualiza a posição.
            posicao += 1


        # Exibe o total.
        print(
            f"Total aguardando: {self.tamanho}"
        )

        print("----------------------------------------\n")


# ==============================================================================
# PARTE 4: EXECUÇÃO E TESTES DO SISTEMA
# ==============================================================================

# Cria uma nova fila de emergência.
emergencia = FilaEmergencia()


# ==============================================================================
# TESTE 1: CHEGADA DOS PACIENTES
# ==============================================================================

print("=== 1. CHEGADA DOS PACIENTES ===")


# João chega primeiro.
# É verde e possui 30 anos.
emergencia.adicionar(
    "João",
    30,
    "verde"
)


# Maria chega depois.
# É amarela e possui 65 anos.
emergencia.adicionar(
    "Maria",
    65,
    "amarelo"
)


# Carlos chega depois.
# É vermelho e possui 40 anos.
emergencia.adicionar(
    "Carlos",
    40,
    "vermelho"
)


# Ana chega depois.
# Também é vermelha, porém possui 70 anos.
#
# Como Ana e Carlos possuem a mesma classificação,
# a idade será utilizada como segundo critério.
emergencia.adicionar(
    "Ana",
    70,
    "vermelho"
)


# Pedro chega depois.
# É amarelo e possui 75 anos.
#
# Como Maria e Pedro são amarelos,
# Pedro ficará antes de Maria por ser mais velho.
emergencia.adicionar(
    "Pedro",
    75,
    "amarelo"
)


# Lucas chega por último.
# É verde e possui 20 anos.
emergencia.adicionar(
    "Lucas",
    20,
    "verde"
)


# Exibe a fila completa.
emergencia.listar_espera()


# ==============================================================================
# TESTE 2: TESTANDO EMPATE DE CLASSIFICAÇÃO E IDADE
# ==============================================================================

print("=== 2. TESTANDO MESMA CLASSIFICAÇÃO E MESMA IDADE ===")


# Rafael possui a mesma classificação e idade de Carlos.
#
# Como Carlos chegou primeiro, ele deverá continuar
# na frente de Rafael.
emergencia.adicionar(
    "Rafael",
    40,
    "vermelho"
)


# Exibe a fila para verificar a ordem.
emergencia.listar_espera()


# ==============================================================================
# TESTE 3: ATENDIMENTO
# ==============================================================================

print("=== 3. ATENDIMENTO DOS PACIENTES ===")


# Ana será atendida primeiro.
#
# Ela é vermelha e possui 70 anos.
# Portanto, possui maior prioridade que Carlos e Rafael.
emergencia.atender()


# Mostra a fila depois do atendimento.
emergencia.listar_espera()


# Carlos será atendido antes de Rafael.
#
# Os dois são vermelhos e possuem 40 anos,
# mas Carlos chegou primeiro.
emergencia.atender()


# Mostra novamente a fila.
emergencia.listar_espera()


# Rafael será atendido agora.
emergencia.atender()


# Mostra a fila restante.
emergencia.listar_espera()


# ==============================================================================
# TESTE 4: ATENDIMENTO DOS PACIENTES AMARELOS
# ==============================================================================

print("=== 4. ATENDIMENTO DOS PACIENTES AMARELOS ===")


# Pedro será atendido antes de Maria,
# pois os dois são amarelos e Pedro é mais velho.
emergencia.atender()


# Maria será atendida depois.
emergencia.atender()


# Mostra a fila restante.
emergencia.listar_espera()


# ==============================================================================
# TESTE 5: ATENDIMENTO DOS PACIENTES VERDES
# ==============================================================================

print("=== 5. ATENDIMENTO DOS PACIENTES VERDES ===")


# João possui 30 anos e chegou antes de Lucas.
emergencia.atender()


# Lucas possui 20 anos.
emergencia.atender()


# ==============================================================================
# TESTE 6: FILA VAZIA
# ==============================================================================

print("=== 6. TESTANDO A FILA VAZIA ===")


# Todos os pacientes já foram atendidos.
# Portanto, a fila está vazia.
emergencia.atender()


# Exibe a situação final da fila.
emergencia.listar_espera()

=== 1. CHEGADA DOS PACIENTES ===
-> João entrou como primeiro paciente da emergência.
-> Maria (AMARELO, 65 anos) entrou no início da fila.
-> Carlos (VERMELHO, 40 anos) entrou no início da fila.
-> Ana (VERMELHO, 70 anos) entrou no início da fila.
-> Pedro (AMARELO, 75 anos) entrou na fila.
-> Lucas (VERDE, 20 anos) entrou na fila.

--- 🏥 FILA DE ESPERA DA EMERGÊNCIA ---
1º -> 🔴 [Ana, 70 anos - VERMELHO]
2º -> 🔴 [Carlos, 40 anos - VERMELHO]
3º -> 🟡 [Pedro, 75 anos - AMARELO]
4º -> 🟡 [Maria, 65 anos - AMARELO]
5º -> 🟢 [João, 30 anos - VERDE]
6º -> 🟢 [Lucas, 20 anos - VERDE]
Total aguardando: 6
----------------------------------------

=== 2. TESTANDO MESMA CLASSIFICAÇÃO E MESMA IDADE ===
-> Rafael (VERMELHO, 40 anos) entrou na fila.

--- 🏥 FILA DE ESPERA DA EMERGÊNCIA ---
1º -> 🔴 [Ana, 70 anos - VERMELHO]
2º -> 🔴 [Carlos, 40 anos - VERMELHO]
3º -> 🔴 [Rafael, 40 anos - VERMELHO]
4º -> 🟡 [Pedro, 75 anos - AMARELO]
5º -> 🟡 [Maria, 65 anos - AMARELO]
6º -> 🟢 [João, 30 anos - VERDE]
7º -> 🟢